# Enhanced ESN Features with Nested Evaluation at a Fixed Temperature

This notebook tests whether richer reservoir-trajectory features improve prediction
of thermal conductivity (`k`) and effusivity (`eff`) at one fixed temperature.

Improvements over Notebook 03:

- baseline-relative ESN sensor inputs;
- original versus enhanced reservoir summaries;
- candidate transient windows (0–1.5 s and 0–5 s);
- state-change, total-variation, rate, saturation and near-zero features;
- early/middle/late trajectory summaries;
- collective reservoir-activity features;
- nested hyperparameter selection inside every outer LORO-CV and LOMO-CV fold.

The outer test fold is never used to select the window, ESN parameters, feature set,
PCA dimension or Ridge regularization. This makes the final out-of-fold metrics a
more defensible estimate than selecting settings once from the full dataset.


# 1. Imports and experiment configuration


In [ ]:
from pathlib import Path
from itertools import product
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import linalg
from scipy.signal import savgol_filter
from sklearn.compose import TransformedTargetRegressor
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
FILE_TITLE = "30C"
RANDOM_STATE = 42

# Load and align the longest required window. Shorter candidates are sliced later.
ANALYSIS_WINDOW = (0.0, 5.0)
EARLY_WINDOW = (0.0, 1.0)
MID_WINDOW = (1.0, 3.0)
LATE_WINDOW = (3.0, 5.0)

# Nested-search settings. This compact grid has 96 combinations per target/fold.
SEARCH_OBJECTIVE = "nrmse_range"  # "nrmse_range" (minimize) or "r2" (maximize)
SEARCH_WINDOWS = (1.5, 5.0)
SEARCH_FEATURE_SETS = ("baseline", "enhanced")
SEARCH_RESERVOIR_SIZES = (30, 50)
SEARCH_LEAK_RATES = (0.1, 0.5)
SEARCH_INPUT_MAGNITUDES = (1.5,)
SEARCH_SPECTRAL_RADII = (0.9, 1.3)
SEARCH_WASHOUTS = (0,)
SEARCH_PCA_COMPONENTS = (10,)
SEARCH_RIDGE_ALPHAS = (1.0, 10.0, 100.0)

# Four inner grouped folds control runtime while preserving group separation.
INNER_SPLITS = 4

cwd = Path.cwd().resolve()
data_candidates = [
    cwd / "data" / "02_preprocessed" / FILE_TITLE,
    cwd.parent / "data" / "02_preprocessed" / FILE_TITLE,
    cwd.parent.parent / "Warmth Sensor Notebook" / "data" / "02_preprocessed" / FILE_TITLE,
]
DATA_PATH = next((path for path in data_candidates if path.is_dir()), None)
if DATA_PATH is None:
    attempted = "\n".join(str(path) for path in data_candidates)
    raise FileNotFoundError(f"Data directory not found. Tried:\n{attempted}")

print("Fixed temperature:", FILE_TITLE)
print("Data path:", DATA_PATH)


# 2. Standard material properties and trial loading

The processed CSV values in `k`, `Mass`, `Volume`, `rho`, and `cp` are deliberately
ignored. After loading the sensor data, the notebook overwrites those fields using
the authoritative table below.

`Mass=1 kg` and `Volume=1 m³` are placeholders and can be updated later.


In [ ]:
STANDARD_PROPERTIES = {
    "ps_foam":   {"k": 0.034, "Mass": 1.0, "Volume": 1.0, "rho": 25.0,   "cp": 1400.0},
    "pu_foam":   {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 30.0,   "cp": 1400.0},
    "cork":      {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 240.0,  "cp": 1800.0},
    "wood":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 700.0,  "cp": 1700.0},
    "pdms":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 970.0,  "cp": 1460.0},
    "gypsum":    {"k": 0.170, "Mass": 1.0, "Volume": 1.0, "rho": 800.0,  "cp": 1090.0},
    "cement":    {"k": 0.290, "Mass": 1.0, "Volume": 1.0, "rho": 1440.0, "cp": 750.0},
    "graphite":  {"k": 100.0, "Mass": 1.0, "Volume": 1.0, "rho": 1820.0, "cp": 710.0},
    "bismuth":   {"k": 8.1,   "Mass": 1.0, "Volume": 1.0, "rho": 9780.0, "cp": 130.0},
    "titanium":  {"k": 21.9,  "Mass": 1.0, "Volume": 1.0, "rho": 4506.0, "cp": 523.0},
    "nickel":    {"k": 90.9,  "Mass": 1.0, "Volume": 1.0, "rho": 8908.0, "cp": 461.0},
    "iron":      {"k": 80.4,  "Mass": 1.0, "Volume": 1.0, "rho": 7874.0, "cp": 449.0},
    "aluminum":  {"k": 237.0, "Mass": 1.0, "Volume": 1.0, "rho": 2700.0, "cp": 897.0},
    "copper":    {"k": 401.0, "Mass": 1.0, "Volume": 1.0, "rho": 8960.0, "cp": 385.0},
}

SAMPLE_ALIASES = {
    "ps": "ps_foam", "ps foam": "ps_foam", "ps_foam": "ps_foam",
    "pu": "pu_foam", "pu foam": "pu_foam", "pu_foam": "pu_foam",
    "cork": "cork", "cork fine": "cork", "cork_fine": "cork",
    "wood": "wood",
    "pdms": "pdms",
    "gypsum": "gypsum",
    "cement": "cement",
    "graphite": "graphite", "carbon": "graphite",
    "bi": "bismuth", "bismuth": "bismuth",
    "ti": "titanium", "titanium": "titanium",
    "ni": "nickel", "nickel": "nickel",
    "fe": "iron", "iron": "iron",
    "al": "aluminum", "aluminum": "aluminum",
    "cu": "copper", "copper": "copper",
}

standard_table = (
    pd.DataFrame.from_dict(STANDARD_PROPERTIES, orient="index")
    .rename_axis("Material")
    .reset_index()
)
display(standard_table)

required = {"Sample", "Trial", "Time", "Primary", "Secondary"}
frames = []

for path in sorted(DATA_PATH.glob("*.csv")):
    frame = pd.read_csv(path)
    missing = required - set(frame.columns)
    if missing:
        warnings.warn(f"Skipping {path.name}; missing {sorted(missing)}")
        continue
    frames.append(frame)

if not frames:
    raise ValueError("No valid trial files were found.")

DATA = pd.concat(frames, ignore_index=True)
DATA = DATA.replace([np.inf, -np.inf], np.nan)
for column in ("Trial", "Time", "Primary", "Secondary"):
    DATA[column] = pd.to_numeric(DATA[column], errors="coerce")
DATA = DATA.dropna(subset=list(required)).copy()
DATA["Trial"] = DATA["Trial"].astype(int)

normalized_sample = (
    DATA["Sample"].astype(str).str.strip().str.lower().str.replace("_", " ")
)
DATA["Sample"] = normalized_sample.map(SAMPLE_ALIASES)
unknown_mask = DATA["Sample"].isna()
if unknown_mask.any():
    unknown = sorted(normalized_sample[unknown_mask].unique())
    raise KeyError(f"No standard-property mapping for samples: {unknown}")

# Overwrite processed-file properties unconditionally.
for property_name in ("k", "Mass", "Volume", "rho", "cp"):
    DATA[property_name] = DATA["Sample"].map(
        lambda sample: STANDARD_PROPERTIES[sample][property_name]
    )

DATA["trial_id"] = (
    DATA["Sample"].astype(str) + "_trial_" + DATA["Trial"].astype(str)
)
DATA["eff"] = np.sqrt(DATA["k"] * DATA["rho"] * DATA["cp"])
DATA = DATA.sort_values(["trial_id", "Time"]).reset_index(drop=True)

# Literature k should be stable by material. Density may vary across measured
# specimens/trials, so the derived eff target is allowed to vary by trial.
k_counts = DATA.groupby("Sample")["k"].nunique()
if (k_counts != 1).any():
    raise ValueError(
        f"k is not constant within: {k_counts[k_counts != 1].index.tolist()}"
    )

summary = (
    DATA.groupby(["trial_id", "Sample", "Trial"], as_index=False)
    .agg(
        n_timesteps=("Time", "size"),
        k=("k", "first"),
        eff=("eff", "first"),
    )
)
print(f"Rows: {len(DATA):,}")
print(f"Trials: {DATA['trial_id'].nunique()}")
print(f"Independent materials: {DATA['Sample'].nunique()}")
display(summary.head())


# 3. Detect contact and align every trial

The elbow is used only to establish a common time origin. Prediction uses the fixed
0–5 second response after contact. Trials are never aligned using `k` or `eff`.


In [ ]:
def find_contact_time(
    trial,
    smooth_window=15,
    polyorder=2,
    threshold_frac=0.30,
    skip_samples=5,
):
    clean = (
        trial[["Time", "Primary"]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
        .sort_values("Time")
        .drop_duplicates("Time")
        .reset_index(drop=True)
    )
    time = clean["Time"].to_numpy(float)
    signal = clean["Primary"].to_numpy(float)
    if len(signal) < skip_samples + 7 or np.any(np.diff(time) <= 0):
        raise ValueError("Insufficient or invalid time samples.")

    work_time = time[skip_samples:]
    work_signal = signal[skip_samples:]
    window = min(int(smooth_window), len(work_signal))
    if window % 2 == 0:
        window -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    if window < minimum:
        raise ValueError("Sequence is too short for smoothing.")

    smooth = savgol_filter(work_signal, window, polyorder, mode="interp")
    derivative = np.gradient(smooth, work_time)
    strongest = int(np.argmin(derivative))
    active = derivative < threshold_frac * derivative[strongest]
    elbow = 0
    for position in range(strongest, -1, -1):
        if not active[position]:
            elbow = position + 1
            break
    return float(work_time[elbow])


In [ ]:
aligned_trials = {}
alignment_rows = []

for trial_id, trial in DATA.groupby("trial_id", sort=False):
    trial = trial.sort_values("Time").drop_duplicates("Time").copy()
    try:
        contact_time = find_contact_time(trial)
    except ValueError as exc:
        warnings.warn(f"Skipping {trial_id}: {exc}")
        continue

    trial["time_from_contact"] = trial["Time"] - contact_time
    start, end = ANALYSIS_WINDOW
    trial = trial[
        trial["time_from_contact"].between(start, end, inclusive="both")
    ].copy()
    if len(trial) < 10:
        warnings.warn(f"Skipping {trial_id}: too few post-contact samples")
        continue
    aligned_trials[trial_id] = trial.reset_index(drop=True)
    alignment_rows.append({
        "trial_id": trial_id,
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "contact_time": contact_time,
        "n_analysis_samples": len(trial),
    })

ALIGNMENT = pd.DataFrame(alignment_rows)
print(f"Aligned trials retained: {len(aligned_trials)}")
display(ALIGNMENT.head())


## Check several alignments visually


In [ ]:
def plot_aligned_trials(material=None, maximum_trials=6):
    selected = list(aligned_trials.values())
    if material is not None:
        selected = [x for x in selected if x["Sample"].iloc[0] == material]
    selected = selected[:maximum_trials]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
    for trial in selected:
        label = trial["trial_id"].iloc[0]
        axes[0].plot(trial["time_from_contact"], trial["Primary"], alpha=0.75, label=label)
        axes[1].plot(trial["time_from_contact"], trial["Secondary"], alpha=0.75, label=label)
    for ax, title in zip(axes, ["Primary", "Secondary"]):
        ax.axvline(0, color="black", linestyle=":")
        ax.set(xlabel="Time from contact (s)", ylabel="Sensor response", title=title)
    axes[1].legend(fontsize=7, bbox_to_anchor=(1.04, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_aligned_trials(material=ALIGNMENT["Sample"].iloc[0])


# 4. Interpretable thermal-response features


In [ ]:
def _smooth(values, window=11, polyorder=2):
    values = np.asarray(values, float)
    selected = min(window, len(values))
    if selected % 2 == 0:
        selected -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    return (
        savgol_filter(values, selected, polyorder, mode="interp")
        if selected >= minimum else values.copy()
    )


def _slope(time, values, window):
    mask = (time >= window[0]) & (time <= window[1])
    if mask.sum() < 3:
        return np.nan
    return float(np.polyfit(time[mask], values[mask], 1)[0])


def extract_thermal_features(trial):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    difference = primary - secondary
    result = {}

    for name, signal in {
        "primary": primary,
        "secondary": secondary,
        "difference": difference,
    }.items():
        change = signal - signal[0]
        rate = np.gradient(signal, time)
        result[f"{name}_final_change"] = float(change[-1])
        result[f"{name}_max_abs_change"] = float(np.max(np.abs(change)))
        result[f"{name}_response_auc"] = float(np.trapezoid(np.abs(change), time))
        result[f"{name}_early_slope"] = _slope(time, signal, EARLY_WINDOW)
        result[f"{name}_mid_slope"] = _slope(time, signal, MID_WINDOW)
        result[f"{name}_late_slope"] = _slope(time, signal, LATE_WINDOW)
        result[f"{name}_max_abs_rate"] = float(np.max(np.abs(rate)))
        result[f"{name}_rate_auc"] = float(np.trapezoid(np.abs(rate), time))

    # Dimensionless/cross-sensor summaries.
    primary_auc = result["primary_response_auc"]
    result["secondary_primary_auc_ratio"] = (
        result["secondary_response_auc"] / primary_auc
        if not np.isclose(primary_auc, 0) else np.nan
    )
    result["initial_sensor_difference"] = float(difference[0])
    result["final_sensor_difference"] = float(difference[-1])
    return result


thermal_rows = []
for trial_id, trial in aligned_trials.items():
    features = extract_thermal_features(trial)
    features.update({
        "trial_id": trial_id,
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "k": float(trial["k"].iloc[0]),
        "eff": float(trial["eff"].iloc[0]),
    })
    thermal_rows.append(features)

THERMAL_FEATURES = pd.DataFrame(thermal_rows)
print(f"Thermal features: {len(THERMAL_FEATURES.columns) - 5}")
display(THERMAL_FEATURES.head())


# 5. Manual ESN reservoir and trajectory features

The reservoir equation comes from the senior's code, but the logistic `TC` readout is
removed. Instead, each trial's post-contact reservoir trajectory is summarized into
one feature vector for regression.

Input scaling is fitted inside each outer training fold. Therefore ESN features must
also be generated inside each fold.


In [ ]:
class ManualReservoir:
    def __init__(
        self,
        res_size=30,
        leak_rate=0.1,
        input_magnitude=1.5,
        spectral_radius=1.3,
        washout=0,
        random_state=42,
    ):
        self.res_size = int(res_size)
        self.leak_rate = float(leak_rate)
        self.input_magnitude = float(input_magnitude)
        self.spectral_radius = float(spectral_radius)
        self.washout = int(washout)
        rng = np.random.default_rng(random_state)
        self.Win = (
            rng.random((self.res_size, 1 + 5)) - 0.5
        ) * self.input_magnitude
        W = rng.random((self.res_size, self.res_size)) - 0.5
        radius = np.max(np.abs(linalg.eigvals(W)))
        if not np.isfinite(radius) or np.isclose(radius, 0):
            raise ValueError("Invalid reservoir spectral radius.")
        self.W = (W / radius.real) * self.spectral_radius

    def run(self, sequence):
        sequence = np.asarray(sequence, float)
        if sequence.ndim != 2 or sequence.shape[1] != 5:
            raise ValueError("Expected five ESN input channels.")
        if len(sequence) <= self.washout:
            raise ValueError("Sequence is shorter than washout.")
        state = np.zeros((self.res_size, 1))
        states = []
        for row in sequence:
            u = row.reshape(-1, 1)
            state = (
                (1 - self.leak_rate) * state
                + self.leak_rate * np.tanh(
                    self.Win @ np.vstack((1.0, u)) + self.W @ state
                )
            )
            states.append(state[:, 0].copy())
        return np.asarray(states)[self.washout:]


def slice_trial(trial, window_end):
    selected = trial[
        trial["time_from_contact"].between(0.0, window_end, inclusive="both")
    ].copy()
    if len(selected) < 8:
        raise ValueError(
            f"Only {len(selected)} samples in the 0–{window_end:g} s window."
        )
    return selected


def raw_esn_input(trial, baseline_relative=True):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    if baseline_relative:
        primary = primary - primary[0]
        secondary = secondary - secondary[0]
    difference = primary - secondary
    primary_rate = np.gradient(primary, time)
    secondary_rate = np.gradient(secondary, time)
    return np.column_stack([
        primary, secondary, difference, primary_rate, secondary_rate
    ])


def _linear_slopes(time, values):
    centered_time = time - np.mean(time)
    denominator = np.sum(centered_time ** 2)
    if denominator <= 0:
        return np.zeros(values.shape[1])
    centered_values = values - np.mean(values, axis=0)
    return np.sum(centered_time[:, None] * centered_values, axis=0) / denominator


def _add_unit_vector(result, prefix, values):
    for unit, value in enumerate(np.asarray(values, float)):
        result[f"{prefix}_{unit}"] = float(value)


def summarize_states(states, time, feature_set="enhanced"):
    states = np.asarray(states, float)
    time = np.asarray(time, float)
    result = {}

    # Original Notebook 03 feature set.
    _add_unit_vector(result, "esn_mean", np.mean(states, axis=0))
    _add_unit_vector(result, "esn_std", np.std(states, axis=0))
    _add_unit_vector(result, "esn_final", states[-1])
    if feature_set == "baseline":
        return result
    if feature_set != "enhanced":
        raise ValueError("feature_set must be baseline or enhanced")

    delta_time = np.gradient(time)
    rates = np.gradient(states, time, axis=0)
    _add_unit_vector(result, "esn_change", states[-1] - states[0])
    _add_unit_vector(
        result, "esn_total_variation", np.sum(np.abs(np.diff(states, axis=0)), axis=0)
    )
    _add_unit_vector(result, "esn_max_abs_rate", np.max(np.abs(rates), axis=0))
    _add_unit_vector(
        result, "esn_saturation_fraction", np.mean(np.abs(states) > 0.95, axis=0)
    )
    _add_unit_vector(
        result, "esn_near_zero_fraction", np.mean(np.abs(states) < 0.05, axis=0)
    )

    # Equal-duration early, middle and late segments adapt to every candidate window.
    edges = np.linspace(time[0], time[-1], 4)
    for segment, (start, end) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (time >= start) & (time <= end if segment == 2 else time < end)
        if mask.sum() < 2:
            continue
        _add_unit_vector(
            result, f"esn_segment_{segment}_mean", np.mean(states[mask], axis=0)
        )
        _add_unit_vector(
            result, f"esn_segment_{segment}_slope", _linear_slopes(time[mask], states[mask])
        )

    # Collective dynamics—the quantities shown in the ESN evolution plot.
    collective = {
        "state_norm": np.linalg.norm(states, axis=1),
        "mean_abs_state": np.mean(np.abs(states), axis=1),
        "mean_state": np.mean(states, axis=1),
        "saturated_unit_fraction": np.mean(np.abs(states) > 0.95, axis=1),
        "step_distance": np.r_[0.0, np.linalg.norm(np.diff(states, axis=0), axis=1)],
    }
    for name, values in collective.items():
        result[f"collective_{name}_mean"] = float(np.mean(values))
        result[f"collective_{name}_max"] = float(np.max(values))
        result[f"collective_{name}_final"] = float(values[-1])
        result[f"collective_{name}_auc"] = float(np.trapezoid(np.abs(values), time))
        result[f"collective_{name}_slope"] = float(_linear_slopes(time, values[:, None])[0])
        result[f"collective_{name}_total_variation"] = float(np.sum(np.abs(np.diff(values))))
    return result


# 6. Candidate models and leakage-safe utilities

The expanded feature table can be much wider than the number of trials. Imputation,
scaling and PCA are therefore fitted inside each training fold before Ridge regression.
The regression target is modeled on a `log1p` scale because `k` and `eff` span orders
of magnitude.


In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    target_range = np.ptp(y_true)
    return {
        "n": len(y_true),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": rmse,
        "nrmse_range": rmse / target_range if target_range > 0 else np.nan,
        "r2": r2_score(y_true, y_pred) if target_range > 0 else np.nan,
        "median_ape_pct": np.median(np.abs((y_true - y_pred) / y_true)) * 100,
    }


def make_regressor(alpha, n_components, n_rows, n_columns):
    components = min(int(n_components), int(n_rows) - 1, int(n_columns))
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=components, random_state=RANDOM_STATE)),
        ("ridge", Ridge(alpha=float(alpha))),
    ]
    return TransformedTargetRegressor(
        regressor=Pipeline(steps),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


def parameter_grid():
    keys = (
        "window_end", "feature_set", "res_size", "leak_rate",
        "input_magnitude", "spectral_radius", "washout",
        "pca_components", "ridge_alpha",
    )
    values = product(
        SEARCH_WINDOWS,
        SEARCH_FEATURE_SETS,
        SEARCH_RESERVOIR_SIZES,
        SEARCH_LEAK_RATES,
        SEARCH_INPUT_MAGNITUDES,
        SEARCH_SPECTRAL_RADII,
        SEARCH_WASHOUTS,
        SEARCH_PCA_COMPONENTS,
        SEARCH_RIDGE_ALPHAS,
    )
    return [dict(zip(keys, combination)) for combination in values]


PARAMETER_GRID = parameter_grid()
print("Nested-search combinations:", len(PARAMETER_GRID))


# 7. Fold-local ESN feature extraction

The input scaler is fitted only on the supplied training IDs. The same fitted scaler
and deterministic reservoir are then applied independently to training and validation
or test trials. Reservoir state is reset to zero at the start of every trial.


In [ ]:
def build_feature_table(train_ids, all_ids, params):
    window_end = params["window_end"]
    sliced = {
        trial_id: slice_trial(aligned_trials[trial_id], window_end)
        for trial_id in all_ids
    }
    scaler = StandardScaler().fit(np.vstack([
        raw_esn_input(sliced[trial_id], baseline_relative=True)
        for trial_id in train_ids
    ]))
    reservoir = ManualReservoir(
        res_size=params["res_size"],
        leak_rate=params["leak_rate"],
        input_magnitude=params["input_magnitude"],
        spectral_radius=params["spectral_radius"],
        washout=params["washout"],
        random_state=RANDOM_STATE,
    )
    rows = []
    for trial_id in all_ids:
        trial = sliced[trial_id]
        sequence = scaler.transform(raw_esn_input(trial, baseline_relative=True))
        states = reservoir.run(sequence)
        time = trial["time_from_contact"].to_numpy(float)[params["washout"]:]
        features = summarize_states(states, time, params["feature_set"])
        features.update({
            "trial_id": trial_id,
            "Sample": trial["Sample"].iloc[0],
            "Trial": int(trial["Trial"].iloc[0]),
            "k": float(trial["k"].iloc[0]),
            "eff": float(trial["eff"].iloc[0]),
        })
        rows.append(features)
    return pd.DataFrame(rows).set_index("trial_id")


def feature_columns(table):
    excluded = {"Sample", "Trial", "k", "eff"}
    return [
        column for column in table.select_dtypes(include=np.number).columns
        if column not in excluded
    ]


# 8. True nested LORO-CV and LOMO-CV

For every outer fold:

1. reserve the outer test group;
2. search all candidate settings using grouped inner validation on outer-training data;
3. choose the best setting from pooled inner out-of-fold predictions;
4. rebuild features and refit using all outer-training trials;
5. predict the untouched outer test group exactly once.

This is computationally heavier because ESN input scaling and feature generation are
repeated within the inner folds rather than calculated once from the full dataset.


In [ ]:
def split_groups(metadata, protocol, inner=False):
    if protocol == "loro":
        groups = metadata["Trial"]
        if inner:
            n_splits = min(INNER_SPLITS, groups.nunique())
            return GroupKFold(n_splits=n_splits).split(metadata, groups=groups)
        return LeaveOneGroupOut().split(metadata, groups=groups)
    if protocol == "lomo":
        groups = metadata["Sample"]
        if inner:
            n_splits = min(INNER_SPLITS, groups.nunique())
            return GroupKFold(n_splits=n_splits).split(metadata, groups=groups)
        return LeaveOneGroupOut().split(metadata, groups=groups)
    raise ValueError("protocol must be loro or lomo")


def objective_value(metrics):
    if SEARCH_OBJECTIVE == "nrmse_range":
        return metrics["nrmse_range"]
    if SEARCH_OBJECTIVE == "r2":
        return metrics["r2"]
    raise ValueError("SEARCH_OBJECTIVE must be nrmse_range or r2")


def nested_evaluate(target, protocol, verbose=True):
    metadata = THERMAL_FEATURES[
        ["trial_id", "Sample", "Trial", "k", "eff"]
    ].reset_index(drop=True)
    outer_rows, prediction_rows, selection_rows = [], [], []

    outer = list(split_groups(metadata, protocol, inner=False))
    for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(outer, start=1):
        outer_train = metadata.iloc[outer_train_idx].reset_index(drop=True)
        outer_test = metadata.iloc[outer_test_idx].reset_index(drop=True)
        train_ids = outer_train["trial_id"].tolist()
        test_ids = outer_test["trial_id"].tolist()
        if verbose:
            held = (
                outer_test["Trial"].unique().tolist()
                if protocol == "loro"
                else outer_test["Sample"].unique().tolist()
            )
            print(f"{protocol.upper()} {target}: outer fold {outer_fold}/{len(outer)}, held out {held}")

        candidate_rows = []
        # Reservoir trajectories do not depend on Ridge alpha. Reuse each
        # fold/ESN/feature-table combination across all alpha candidates.
        feature_cache = {}
        inner_indices = list(split_groups(outer_train, protocol, inner=True))
        for candidate_number, params in enumerate(PARAMETER_GRID, start=1):
            inner_predictions = []
            for inner_fold, (inner_train_idx, inner_valid_idx) in enumerate(inner_indices, start=1):
                inner_train = outer_train.iloc[inner_train_idx]
                inner_valid = outer_train.iloc[inner_valid_idx]
                inner_train_ids = inner_train["trial_id"].tolist()
                inner_valid_ids = inner_valid["trial_id"].tolist()
                structural_key = (
                    inner_fold,
                    params["window_end"], params["feature_set"],
                    params["res_size"], params["leak_rate"],
                    params["input_magnitude"], params["spectral_radius"],
                    params["washout"],
                )
                if structural_key not in feature_cache:
                    feature_cache[structural_key] = build_feature_table(
                        inner_train_ids,
                        inner_train_ids + inner_valid_ids,
                        params,
                    )
                table = feature_cache[structural_key]
                columns = feature_columns(table)
                X_train = table.loc[inner_train_ids, columns].reset_index(drop=True)
                X_valid = table.loc[inner_valid_ids, columns].reset_index(drop=True)
                y_train = table.loc[inner_train_ids, target].reset_index(drop=True)
                y_valid = table.loc[inner_valid_ids, target].reset_index(drop=True)
                model = make_regressor(
                    params["ridge_alpha"], params["pca_components"],
                    len(X_train), len(columns),
                )
                model.fit(X_train, y_train)
                prediction = np.clip(
                    model.predict(X_valid), y_train.min(), y_train.max()
                )
                inner_predictions.append(pd.DataFrame({
                    "y_true": y_valid.to_numpy(), "y_pred": prediction,
                }))
            inner_oof = pd.concat(inner_predictions, ignore_index=True)
            metrics = regression_metrics(inner_oof["y_true"], inner_oof["y_pred"])
            candidate_rows.append({
                "candidate": candidate_number, **params, **metrics,
                "selection_score": objective_value(metrics),
            })

        candidates = pd.DataFrame(candidate_rows)
        ascending = SEARCH_OBJECTIVE == "nrmse_range"
        candidates = candidates.sort_values(
            "selection_score", ascending=ascending, na_position="last"
        ).reset_index(drop=True)
        best = candidates.iloc[0].to_dict()
        best_params = {key: best[key] for key in PARAMETER_GRID[0]}
        selection_rows.append({
            "outer_fold": outer_fold, "protocol": protocol, "target": target,
            **best_params, "inner_selection_score": best["selection_score"],
        })

        final_table = build_feature_table(
            train_ids, train_ids + test_ids, best_params
        )
        columns = feature_columns(final_table)
        X_train = final_table.loc[train_ids, columns].reset_index(drop=True)
        X_test = final_table.loc[test_ids, columns].reset_index(drop=True)
        y_train = final_table.loc[train_ids, target].reset_index(drop=True)
        y_test = final_table.loc[test_ids, target].reset_index(drop=True)
        final_model = make_regressor(
            best_params["ridge_alpha"], best_params["pca_components"],
            len(X_train), len(columns),
        )
        final_model.fit(X_train, y_train)
        prediction = np.clip(
            final_model.predict(X_test), y_train.min(), y_train.max()
        )
        outer_rows.append({
            "outer_fold": outer_fold, "protocol": protocol, "target": target,
            "held_out_group": ", ".join(
                str(value) for value in (
                    outer_test["Trial"].unique()
                    if protocol == "loro" else outer_test["Sample"].unique()
                )
            ),
            **best_params, **regression_metrics(y_test, prediction),
        })
        prediction_rows.append(pd.DataFrame({
            "trial_id": test_ids,
            "Sample": outer_test["Sample"].to_numpy(),
            "Trial": outer_test["Trial"].to_numpy(),
            "outer_fold": outer_fold,
            "y_true": y_test.to_numpy(),
            "y_pred": prediction,
        }))

    folds = pd.DataFrame(outer_rows)
    predictions = pd.concat(prediction_rows, ignore_index=True)
    selections = pd.DataFrame(selection_rows)
    overall = pd.DataFrame([{
        "protocol": protocol, "target": target,
        **regression_metrics(predictions["y_true"], predictions["y_pred"]),
    }])
    return folds, overall, predictions, selections


# 9. Run the nested evaluations

This cell is intentionally expensive. With the default grid it evaluates 96 parameter
combinations inside every outer fold for both targets and protocols. Start with one
target/protocol if you want to estimate runtime, then enable all four runs.


In [ ]:
RUNS = [
    ("k", "loro"),
    ("eff", "loro"),
    ("k", "lomo"),
    ("eff", "lomo"),
]

NESTED_RESULTS = []
NESTED_FOLDS = {}
NESTED_PREDICTIONS = {}
NESTED_SELECTIONS = {}

for target, protocol in RUNS:
    folds, overall, predicted, selections = nested_evaluate(target, protocol)
    NESTED_RESULTS.append(overall)
    NESTED_FOLDS[(protocol, target)] = folds
    NESTED_PREDICTIONS[(protocol, target)] = predicted
    NESTED_SELECTIONS[(protocol, target)] = selections

NESTED_RESULTS = pd.concat(NESTED_RESULTS, ignore_index=True)
display(NESTED_RESULTS)


# 10. Selected settings and fold-level performance


In [ ]:
for target, protocol in RUNS:
    print(f"\n{protocol.upper()} → {target}: selected settings by outer fold")
    display(NESTED_SELECTIONS[(protocol, target)])
    print("Outer-fold metrics:")
    display(NESTED_FOLDS[(protocol, target)])


# 11. Actual-versus-predicted plots


In [ ]:
def plot_actual_predicted(predictions, title):
    shown = predictions[
        (predictions["y_true"] > 0) & (predictions["y_pred"] > 0)
    ]
    fig, ax = plt.subplots(figsize=(7, 6))
    for material, part in shown.groupby("Sample"):
        ax.scatter(part["y_true"], part["y_pred"], label=material, alpha=0.75)
    low = min(shown["y_true"].min(), shown["y_pred"].min())
    high = max(shown["y_true"].max(), shown["y_pred"].max())
    ax.plot([low, high], [low, high], "k:", label="Ideal")
    ax.set(xscale="log", yscale="log", xlabel="Actual", ylabel="Predicted", title=title)
    ax.legend(fontsize=7, bbox_to_anchor=(1.04, 1), loc="upper left")
    fig.tight_layout()
    plt.show()


for target, protocol in RUNS:
    plot_actual_predicted(
        NESTED_PREDICTIONS[(protocol, target)],
        f"Enhanced ESN: nested {protocol.upper()} prediction of {target} at {FILE_TITLE}",
    )


# 12. Interpretation and reporting rules

- Compare these nested results with Notebook 03 using the same temperature and target.
- A selected 1.5 s window supports the early-transient hypothesis; a selected 5 s
  window indicates that later settling behavior adds predictive information.
- Count how often `enhanced` versus `baseline` is selected across outer folds rather
  than interpreting one fold in isolation.
- LORO-CV measures repeatability for represented materials.
- LOMO-CV measures generalization to a material absent from training.
- Report the combined outer out-of-fold metrics as the principal result. Inner scores
  are used only for model selection.
- Per-material LOMO fold R² is undefined because all trials of one material share one
  target value; use overall out-of-fold R² and per-material MAE/percentage error.
- More features do not guarantee improvement. If the enhanced set is rarely selected
  or worsens outer performance, retain the simpler baseline.
- The reservoir is a fixed nonlinear feature generator; Ridge is the trained readout.
